In [2]:
"""
fetch_historical_data.py
========================
Fetches historical OHLCV bars from MetaTrader 5 for any symbol and
date range, then saves them to a clean CSV ready for the training pipeline.

Run this ONCE per symbol/timeframe you want to train on.
MT5 must be open and logged in (live or demo account) when you run this.

Requirements:
    pip install MetaTrader5 pandas pytz

Usage (Jupyter):
    Just run the cell. Edit the CONFIG block below.

Usage (CLI):
    python fetch_historical_data.py
"""

from datetime import datetime
import sys
import warnings

import pandas as pd
import pytz

warnings.filterwarnings("ignore")

# =============================================================================
#  CONFIG — edit these values
# =============================================================================
SYMBOL      = "XAUUSD"        # exact symbol name as shown in MT5 Market Watch
TIMEFRAME   = "H1"              # M1 M5 M15 M30 H1 H4 D1
DATE_FROM   = datetime(2020, 1,  1)   # start of historical range
DATE_TO     = datetime(2026, 1,  1)   # end   of historical range (exclusive)
OUTPUT_CSV  = "XAUUSD_H1.csv"         # output filename (saved in working directory)
TIMEZONE    = "Etc/UTC"               # keep UTC — pipeline expects UTC timestamps
# =============================================================================

# Timeframe string -> MT5 constant name
TF_MAP = {
    "M1":  "TIMEFRAME_M1",
    "M5":  "TIMEFRAME_M5",
    "M15": "TIMEFRAME_M15",
    "M30": "TIMEFRAME_M30",
    "H1":  "TIMEFRAME_H1",
    "H4":  "TIMEFRAME_H4",
    "D1":  "TIMEFRAME_D1",
}

def fetch():
    try:
        import MetaTrader5 as mt5
    except ImportError:
        sys.exit("[ERROR] MetaTrader5 package not installed.\n"
                 "        Run:  pip install MetaTrader5")

    print(f"MetaTrader5 package  v{mt5.__version__}  by {mt5.__author__}")

    # ── Initialise ───────────────────────────────────────────────────────────
    if not mt5.initialize():
        sys.exit(f"[ERROR] mt5.initialize() failed: {mt5.last_error()}")
    print(f"[MT5]  Connected — build {mt5.version()}")

    # ── Select symbol ────────────────────────────────────────────────────────
    if not mt5.symbol_select(SYMBOL, True):
        mt5.shutdown()
        sys.exit(f"[ERROR] Symbol '{SYMBOL}' not found in Market Watch.\n"
                 "        Check the exact name (e.g. XAUUSD vs XAUUSD.m vs XAUUSDm)")

    info = mt5.symbol_info(SYMBOL)
    point = info.point if info else 0.01
    print(f"[MT5]  Symbol: {SYMBOL}  Point: {point}  Digits: {info.digits if info else '?'}")

    # ── Validate timeframe ───────────────────────────────────────────────────
    tf_attr = TF_MAP.get(TIMEFRAME.upper())
    if tf_attr is None:
        mt5.shutdown()
        sys.exit(f"[ERROR] Unknown timeframe '{TIMEFRAME}'. Choose from: {list(TF_MAP)}")
    tf = getattr(mt5, tf_attr)

    # ── Build UTC-aware date range ───────────────────────────────────────────
    tz      = pytz.timezone(TIMEZONE)
    utc_from = tz.localize(DATE_FROM)
    utc_to   = tz.localize(DATE_TO)
    print(f"[MT5]  Requesting {TIMEFRAME} bars from {utc_from.date()} to {utc_to.date()} ...")

    # ── Fetch ────────────────────────────────────────────────────────────────
    rates = mt5.copy_rates_range(SYMBOL, tf, utc_from, utc_to)
    mt5.shutdown()

    if rates is None or len(rates) == 0:
        sys.exit("[ERROR] No bars returned. Possible causes:\n"
                 "  - Date range has no data for this symbol on this broker\n"
                 "  - Symbol requires a different name (try without .m suffix)\n"
                 "  - MT5 history for this period not downloaded yet\n"
                 "    (Open the chart in MT5 and scroll back to force download)")

    # ── Build DataFrame ──────────────────────────────────────────────────────
    df = pd.DataFrame(rates)
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df.set_index("time", inplace=True)

    # Rename tick_volume -> volume, drop spread column if present
    df.rename(columns={"tick_volume": "volume", "real_volume": "real_vol"}, inplace=True)
    keep = [c for c in ["open", "high", "low", "close", "volume"] if c in df.columns]
    df = df[keep].astype(float)

    # ── Quality report ───────────────────────────────────────────────────────
    n_bars     = len(df)
    date_start = df.index[0].strftime("%Y-%m-%d %H:%M")
    date_end   = df.index[-1].strftime("%Y-%m-%d %H:%M")
    price_min  = df["close"].min()
    price_max  = df["close"].max()
    atr_proxy  = (df["high"] - df["low"]).mean()
    gaps       = df.index.to_series().diff().dropna()
    expected   = gaps.mode()[0]
    gap_bars   = (gaps > expected * 1.5).sum()

    print(f"\n{'='*55}")
    print(f"  Data Quality Report")
    print(f"{'='*55}")
    print(f"  Bars fetched   : {n_bars:,}")
    print(f"  Date range     : {date_start}  ->  {date_end}")
    print(f"  Price range    : {price_min:.2f}  ->  {price_max:.2f}")
    print(f"  Mean ATR (H-L) : {atr_proxy:.2f}")
    print(f"  Missing bars   : {gap_bars:,}  (weekend/holiday gaps expected)")
    print(f"{'='*55}\n")

    if n_bars < 1000:
        print(f"[WARN]  Only {n_bars} bars fetched. Consider extending the date range.\n"
              "        Training needs at least 5,000+ bars for meaningful results.")

    # ── Save ─────────────────────────────────────────────────────────────────
    df.to_csv(OUTPUT_CSV)
    print(f"[OK]   Saved {n_bars:,} bars to '{OUTPUT_CSV}'")
    print(f"       File size: {pd.io.common.get_handle(OUTPUT_CSV,'r').handle.seek(0,2) if False else ''}"
          f"{round(Path(OUTPUT_CSV).stat().st_size / 1024, 1)} KB")
    print(f"\nNext step: set csv_file = '{OUTPUT_CSV}' in the training pipeline.")
    return df


# ── Entrypoint ────────────────────────────────────────────────────────────────
from pathlib import Path

if __name__ == "__main__":
    fetch()
else:
    # When imported or run as a Jupyter cell, execute immediately
    fetch()

MetaTrader5 package  v5.0.5572  by MetaQuotes Ltd.
[MT5]  Connected — build (500, 5833, '25 Apr 2026')
[MT5]  Symbol: XAUUSD  Point: 0.001  Digits: 3
[MT5]  Requesting H1 bars from 2020-01-01 to 2026-01-01 ...

  Data Quality Report
  Bars fetched   : 13,584
  Date range     : 2023-09-04 01:00  ->  2025-12-31 23:00
  Price range    : 1815.03  ->  4546.69
  Mean ATR (H-L) : 8.17
  Missing bars   : 709  (weekend/holiday gaps expected)

[OK]   Saved 13,584 bars to 'XAUUSD_H1.csv'
       File size: 873.9 KB

Next step: set csv_file = 'XAUUSD_H1.csv' in the training pipeline.


In [1]:
"""
SMC AI Filter — XGBoost Training & ONNX Export Pipeline
========================================================
Workflow:
  1.  Load OHLCV data from a CSV exported by fetch_historical_data.py
      (or live from MT5 directly, or synthetic for quick tests)
  2.  Reconstruct SMC signal events (OB / FVG / BOS) — same logic as the EA
  3.  Label each event: 1 = TP hit, 0 = SL hit (within 2000-bar lookahead)
  4.  Engineer 12 normalised features per signal event
  5.  Train an XGBoost binary classifier with 5-fold cross-validation
  6.  Export the full pipeline to ONNX via onnxmltools
  7.  Verify the ONNX model with onnxruntime
  8.  Save a feature CSV for auditing

Requirements:
    pip install pandas numpy scikit-learn xgboost skl2onnx onnxmltools onnxruntime pytz
    pip install MetaTrader5   (only needed if using data_source="mt5")

Recommended workflow:
    Step 1 — run fetch_historical_data.py  to produce  XAUUSD_H1.csv
    Step 2 — set  DATA_SOURCE = "csv"  and  CSV_FILE = "XAUUSD_H1.csv"  below
    Step 3 — run this script
    Step 4 — copy smc_filter.onnx  ->  MQL5/Files/
    Step 5 — attach SMC_AI_Filter.mq5 to chart, run Strategy Tester
"""

import argparse
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =============================================================================
#  NOTEBOOK CONFIG — edit these values (ignored when run from CLI)
# =============================================================================
NOTEBOOK_CONFIG = dict(
    # ── Data source ──────────────────────────────────────────────────────────
    # "csv"     : load from CSV_FILE produced by fetch_historical_data.py  ✅ recommended
    # "mt5"     : pull bars live from MT5 (MT5 must be open)
    # "synthetic": random-walk data for quick pipeline tests
    data_source = "csv",
    csv_file    = "XAUUSD_H1.csv",   # path to the CSV file

    # ── MT5 live-pull settings (used only when data_source="mt5") ────────────
    symbol    = "XAUUSD.m",
    timeframe = "H1",
    bars      = 50000,

    # ── Trade parameters — MUST match EA inputs exactly ──────────────────────
    # XAUUSD point = 0.01, so:
    #   sl_pts=3500  ->  SL distance = 3500 × 0.01 = $35.00
    #   tp_pts=7500  ->  TP distance = 7500 × 0.01 = $75.00
    sl_pts    = 3500,
    tp_pts    = 7500,

    # ── SMC detection parameters — MUST match EA inputs exactly ─────────────
    swing_len = 5,       # SwingPeriod in EA
    fvg_min   = 3,       # FVG_MinPoints in EA
    fib_lvl   = 61.8,   # Fib_Trade_lvls in EA

    # ── Output ───────────────────────────────────────────────────────────────
    out_dir   = ".",     # where to save smc_features.csv (audit log)

    # Direct path to MT5 Files folder -- model is saved here automatically.
    # MT5 -> File -> Open Data Folder -> MQL5 -> Files
    mt5_files_path = r'C:\Users\Honey\AppData\Roaming\MetaQuotes\Terminal\776D2ACDFA4F66FAF3C8985F75FA9FF6\MQL5\Files',
)
# =============================================================================


# ---------------------------------------------------------------------------
# Symbol point-size lookup  (mirrors MQL5 _Point behaviour)
# ---------------------------------------------------------------------------
SYMBOL_POINT = {
    "XAUUSD": 0.01,
    "XAGUSD": 0.001,
    "USDJPY": 0.001,
    "EURJPY": 0.001,
    "GBPJPY": 0.001,
    "US30":   0.01,
    "NAS100": 0.01,
    "SPX500": 0.01,
    "USTEC":  0.01,
}

def get_point(symbol: str) -> float:
    sym = symbol.upper()
    for key, val in SYMBOL_POINT.items():
        if key in sym:
            return val
    return 0.00001    # default: standard 5-digit FX


# ---------------------------------------------------------------------------
# Config resolution (Jupyter-safe argparse)
# ---------------------------------------------------------------------------
def get_config():
    """
    Returns a plain Namespace from NOTEBOOK_CONFIG when inside Jupyter,
    or from CLI arguments when run from a terminal.
    """
    running_in_jupyter = (
        "ipykernel_launcher" in sys.argv[0]
        or any(a.startswith("--f=") or a == "-f" for a in sys.argv[1:])
    )

    if running_in_jupyter:
        print("[Config] Jupyter detected -- using NOTEBOOK_CONFIG.")
        print("[Config] Edit the NOTEBOOK_CONFIG dict at the top of this file.\n")
        return argparse.Namespace(**NOTEBOOK_CONFIG)

    p = argparse.ArgumentParser(description="SMC XGBoost -> ONNX trainer")
    p.add_argument("--data_source", default="csv",
                   choices=["csv", "mt5", "synthetic"])
    p.add_argument("--csv_file",   default="XAUUSD_H1.csv")
    p.add_argument("--symbol",     default="XAUUSD.m")
    p.add_argument("--timeframe",  default="H1")
    p.add_argument("--bars",       type=int,   default=50000)
    p.add_argument("--sl_pts",     type=int,   default=3500)
    p.add_argument("--tp_pts",     type=int,   default=7500)
    p.add_argument("--swing_len",  type=int,   default=5)
    p.add_argument("--fvg_min",    type=int,   default=3)
    p.add_argument("--fib_lvl",    type=float, default=61.8)
    p.add_argument("--out_dir",    default=".")
    p.add_argument("--mt5_files_path", default="",
                   help="Direct path to MQL5/Files folder")
    return p.parse_args()


# ---------------------------------------------------------------------------
# Data loaders
# ---------------------------------------------------------------------------
def load_csv(csv_file: str) -> pd.DataFrame:
    """
    Load the CSV produced by fetch_historical_data.py.
    Handles both UTC-aware and naive timestamps.
    Expected columns: time (index), open, high, low, close, volume
    """
    if not Path(csv_file).exists():
        sys.exit(
            f"[ERROR] CSV file not found: '{csv_file}'\n"
            "        Run fetch_historical_data.py first to generate it."
        )

    df = pd.read_csv(csv_file, index_col=0, parse_dates=True)

    # Normalise index to UTC-aware
    if df.index.tz is None:
        df.index = pd.to_datetime(df.index, utc=True)
    else:
        df.index = df.index.tz_convert("UTC")

    df.index.name = "time"

    # Keep only OHLCV; rename tick_volume if present
    df.rename(columns={"tick_volume": "volume"}, inplace=True)
    required = ["open", "high", "low", "close"]
    missing  = [c for c in required if c not in df.columns]
    if missing:
        sys.exit(f"[ERROR] CSV is missing columns: {missing}\n"
                 f"        Found columns: {list(df.columns)}")

    if "volume" not in df.columns:
        df["volume"] = 1.0   # placeholder if volume absent

    df = df[["open", "high", "low", "close", "volume"]].astype(float)
    df.dropna(inplace=True)
    df.sort_index(inplace=True)

    date_start = df.index[0].strftime("%Y-%m-%d")
    date_end   = df.index[-1].strftime("%Y-%m-%d")
    atr_proxy  = (df["high"] - df["low"]).mean()

    print(f"[CSV]   Loaded {len(df):,} bars from '{csv_file}'")
    print(f"        Date range : {date_start}  ->  {date_end}")
    print(f"        Price range: {df['close'].min():.2f}  ->  {df['close'].max():.2f}")
    print(f"        Mean H-L   : {atr_proxy:.4f}  ({atr_proxy:.2f} for XAUUSD-style)\n")
    return df


def load_mt5_bars(symbol: str, tf_str: str, n: int) -> pd.DataFrame:
    try:
        import MetaTrader5 as mt5
    except ImportError:
        sys.exit("MetaTrader5 package not installed.  pip install MetaTrader5")

    if not mt5.initialize():
        sys.exit(f"mt5.initialize() failed: {mt5.last_error()}")

    tf = getattr(mt5, f"TIMEFRAME_{tf_str}", None)
    if tf is None:
        mt5.shutdown()
        sys.exit(f"Unknown timeframe '{tf_str}'")

    rates = mt5.copy_rates_from_pos(symbol, tf, 0, n)
    mt5.shutdown()

    if rates is None or len(rates) == 0:
        sys.exit("No data returned from MT5. Check symbol name and date availability.")

    df = pd.DataFrame(rates)
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df.rename(columns={"tick_volume": "volume"}, inplace=True)
    df.set_index("time", inplace=True)
    df = df[["open", "high", "low", "close", "volume"]].astype(float)
    print(f"[MT5]   Loaded {len(df):,} bars of {symbol} {tf_str}")
    return df


def make_synthetic_bars(n: int = 20000, symbol: str = "XAUUSD",
                         timeframe: str = "H1") -> pd.DataFrame:
    """Realistic random-walk data — for quick pipeline tests only."""
    np.random.seed(42)
    sym = symbol.upper()
    freq_map = {"M1":"1min","M5":"5min","M15":"15min","M30":"30min",
                "H1":"1h","H4":"4h","D1":"1D"}
    freq = freq_map.get(timeframe, "1h")
    dates = pd.date_range("2020-01-01", periods=n, freq=freq, tz="UTC")

    if   "XAU" in sym: base, step, noise, wick = 1900.0, 0.80, 2.50, 4.00
    elif "XAG" in sym: base, step, noise, wick = 24.0,   0.05, 0.15, 0.25
    elif "JPY" in sym: base, step, noise, wick = 130.0,  0.05, 0.15, 0.25
    elif "US30" in sym: base, step, noise, wick = 34000., 50., 120., 200.
    else:              base, step, noise, wick = 1.1,  3e-4, 5e-4, 8e-4

    close     = base + np.cumsum(np.random.normal(0, step, n))
    close     = np.maximum(close, base * 0.5)
    body      = np.abs(np.random.normal(0, noise, n))
    wu        = np.abs(np.random.normal(0, wick,  n))
    wd        = np.abs(np.random.normal(0, wick,  n))
    direction = np.sign(np.random.normal(0, 1, n))
    open_     = close - direction * body
    high      = np.maximum(close, open_) + wu
    low       = np.minimum(close, open_) - wd
    volume    = np.random.randint(500, 8000, n).astype(float)

    df = pd.DataFrame({"open": open_, "high": high, "low": low,
                        "close": close, "volume": volume}, index=dates)
    print(f"[Synth] Generated {n:,} synthetic {symbol} {timeframe} bars "
          f"({close.min():.2f} – {close.max():.2f})")
    return df


# ---------------------------------------------------------------------------
# Technical indicators  (no TA-Lib dependency)
# ---------------------------------------------------------------------------
def rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain  = delta.clip(lower=0).ewm(com=period-1, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(com=period-1, adjust=False).mean()
    return (100 - 100 / (1 + gain / loss.replace(0, np.nan))).fillna(50)


def atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - df["close"].shift(1)).abs(),
        (df["low"]  - df["close"].shift(1)).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(com=period-1, adjust=False).mean()


def adx(df: pd.DataFrame, period: int = 14) -> pd.Series:
    up, down = df["high"].diff(), -df["low"].diff()
    pdm = up.where((up > down) & (up > 0), 0.0)
    ndm = down.where((down > up) & (down > 0), 0.0)
    tr_s = atr(df, period)
    pdi  = 100 * pdm.ewm(com=period-1, adjust=False).mean() / tr_s.replace(0, np.nan)
    ndi  = 100 * ndm.ewm(com=period-1, adjust=False).mean() / tr_s.replace(0, np.nan)
    dx   = (100 * (pdi - ndi).abs() / (pdi + ndi).replace(0, np.nan)).fillna(0)
    return dx.ewm(com=period-1, adjust=False).mean().fillna(20)


# ---------------------------------------------------------------------------
# SMC signal detection  (mirrors EA logic exactly)
# ---------------------------------------------------------------------------
SIGNAL_OB  = 0
SIGNAL_FVG = 1
SIGNAL_BOS = 2


def _is_swing_high(high: np.ndarray, idx: int, length: int) -> bool:
    n = len(high)
    for i in range(1, length + 1):
        l, r = idx + i, idx - i
        if r < 0:                                  return False
        if high[idx] <= high[r]:                   return False
        if l < n and high[idx] < high[l]:          return False
    return True


def _is_swing_low(low: np.ndarray, idx: int, length: int) -> bool:
    n = len(low)
    for i in range(1, length + 1):
        l, r = idx + i, idx - i
        if r < 0:                                  return False
        if low[idx] >= low[r]:                     return False
        if l < n and low[idx] > low[l]:            return False
    return True


def detect_signals(df: pd.DataFrame, args) -> pd.DataFrame:
    point   = get_point(args.symbol)
    sw_len  = args.swing_len
    fvg_min = args.fvg_min * point

    H = df["high"].values
    L = df["low"].values
    O = df["open"].values
    C = df["close"].values
    n = len(df)
    events = []

    # ── Order Blocks ──────────────────────────────────────────────────────────
    for i in range(4, n - 4):
        # Bullish OB: bearish candle at [i+3] before bullish impulse
        if (O[i+3] > C[i+3] and O[i+2] < C[i+2] and
                O[i] < C[i] and O[i+3] < C[i+2]):
            events.append(dict(bar_idx=i, signal_type=SIGNAL_OB, direction=1,
                               zone_high=H[i+3], zone_low=L[i+3],
                               entry_price=(H[i+3]+L[i+3])/2))
        # Bearish OB: bullish candle at [i+3] before bearish impulse
        if (O[i+3] < C[i+3] and O[i+2] > C[i+2] and
                O[i] > C[i] and O[i+3] > C[i+2]):
            events.append(dict(bar_idx=i, signal_type=SIGNAL_OB, direction=-1,
                               zone_high=H[i+3], zone_low=L[i+3],
                               entry_price=(H[i+3]+L[i+3])/2))

    # ── Fair Value Gaps ───────────────────────────────────────────────────────
    for i in range(2, n - 2):
        la, ha = L[i+2], H[i+2]
        hc, lc = H[i],   L[i]
        if la > hc and (la - hc) >= fvg_min:
            events.append(dict(bar_idx=i, signal_type=SIGNAL_FVG, direction=1,
                               zone_high=la, zone_low=hc,
                               entry_price=(la+hc)/2))
        elif ha < lc and (lc - ha) >= fvg_min:
            events.append(dict(bar_idx=i, signal_type=SIGNAL_FVG, direction=-1,
                               zone_high=lc, zone_low=ha,
                               entry_price=(lc+ha)/2))

    # ── Break of Structure ────────────────────────────────────────────────────
    sh_list = [i for i in range(sw_len, n-sw_len) if _is_swing_high(H, i, sw_len)]
    sl_list = [i for i in range(sw_len, n-sw_len) if _is_swing_low(L,  i, sw_len)]

    for sl_idx in sl_list:      # BOS Buy: break below swing low
        for j in range(sl_idx+1, min(sl_idx+50, n)):
            if L[j] < L[sl_idx]:
                events.append(dict(bar_idx=j, signal_type=SIGNAL_BOS, direction=1,
                                   zone_high=L[sl_idx]+fvg_min,
                                   zone_low =L[sl_idx]-fvg_min,
                                   entry_price=L[sl_idx]))
                break

    for sh_idx in sh_list:      # BOS Sell: break above swing high
        for j in range(sh_idx+1, min(sh_idx+50, n)):
            if H[j] > H[sh_idx]:
                events.append(dict(bar_idx=j, signal_type=SIGNAL_BOS, direction=-1,
                                   zone_high=H[sh_idx]+fvg_min,
                                   zone_low =H[sh_idx]-fvg_min,
                                   entry_price=H[sh_idx]))
                break

    sig_df = (pd.DataFrame(events)
                .drop_duplicates(subset=["bar_idx","signal_type","direction"])
                .sort_values("bar_idx")
                .reset_index(drop=True))

    print(f"[SMC]   Detected {len(sig_df):,} raw signal events  "
          f"(OB={(sig_df.signal_type==SIGNAL_OB).sum()}, "
          f"FVG={(sig_df.signal_type==SIGNAL_FVG).sum()}, "
          f"BOS={(sig_df.signal_type==SIGNAL_BOS).sum()})")
    return sig_df


# ---------------------------------------------------------------------------
# Labeller: walk-forward TP/SL simulation
# ---------------------------------------------------------------------------
def label_events(df: pd.DataFrame, sig_df: pd.DataFrame, args) -> pd.DataFrame:
    """
    Simulate each detected signal forward bar-by-bar.
    Label = 1 if TP is hit first, 0 if SL is hit first.
    Events where neither is hit within 2000 bars are dropped (expired).

    SL / TP are in EA 'points' — converted to price distance via get_point().
    """
    point    = get_point(args.symbol)
    sl_dist  = args.sl_pts * point
    tp_dist  = args.tp_pts * point
    H        = df["high"].values
    L        = df["low"].values
    n_bars   = len(df)
    LOOKFWD  = 2000

    print(f"[Label] Symbol: {args.symbol}  Point: {point}")
    print(f"[Label] SL: {args.sl_pts} pts = {sl_dist:.4f}  |  "
          f"TP: {args.tp_pts} pts = {tp_dist:.4f}  |  Lookahead: {LOOKFWD} bars")

    # Auto-scale safety net: if SL > 10× mean bar range, trades will rarely resolve
    mean_hl = float((df["high"] - df["low"]).mean())
    if sl_dist > 10 * mean_hl:
        sl_dist = round(2.0 * mean_hl, 5)
        tp_dist = round(4.0 * mean_hl, 5)
        print(f"[Label] WARN: SL/TP too large for data volatility.")
        print(f"[Label] Auto-scaled -> SL={sl_dist:.4f}  TP={tp_dist:.4f} (2x/4x mean H-L)")
        print(f"[Label] NOTE: This only applies to synthetic data. "
              "Real CSV data will use your exact sl_pts/tp_pts.")

    labels = []
    for _, row in sig_df.iterrows():
        entry    = float(row["entry_price"])
        start    = int(row["bar_idx"])
        d        = int(row["direction"])
        sl_price = entry - d * sl_dist
        tp_price = entry + d * tp_dist
        outcome  = np.nan

        for j in range(start + 1, min(start + LOOKFWD, n_bars)):
            if d == 1:    # buy
                if L[j] <= sl_price: outcome = 0; break
                if H[j] >= tp_price: outcome = 1; break
            else:         # sell
                if H[j] >= sl_price: outcome = 0; break
                if L[j] <= tp_price: outcome = 1; break

        labels.append(outcome)

    result = sig_df.copy()
    result["label"] = labels
    result.dropna(subset=["label"], inplace=True)
    result["label"] = result["label"].astype(int)

    expired  = len(sig_df) - len(result)
    win_rate = result["label"].mean()
    rr       = args.tp_pts / args.sl_pts

    print(f"\n[Label] Results:")
    print(f"        Total signals  : {len(sig_df):,}")
    print(f"        Labelled       : {len(result):,}  ({100*len(result)/len(sig_df):.1f}% resolved)")
    print(f"        Expired (NaN)  : {expired:,}")
    print(f"        Win rate       : {win_rate:.1%}")
    print(f"        TP hits        : {result['label'].sum():,}")
    print(f"        SL hits        : {(result['label']==0).sum():,}")
    print(f"        Risk:Reward    : 1:{rr:.2f}")
    print(f"        Expected win%  : {100/(1+rr):.1f}%  (random baseline)\n")
    return result


# ---------------------------------------------------------------------------
# Feature engineering  (12 features — must match MQL5 RunONNX exactly)
# ---------------------------------------------------------------------------
FEATURE_COLS = [
    "f_signal_ob",          # one-hot: is OB signal?
    "f_signal_fvg",         # one-hot: is FVG signal?
    "f_signal_bos",         # one-hot: is BOS signal?
    "f_direction",          # +1 bullish / -1 bearish
    "f_zone_width_atr",     # zone width normalised by ATR(14)
    "f_dist_to_zone_atr",   # |entry - zone midpoint| / ATR(14)
    "f_fib_pct",            # retrace depth inside zone [0,1]
    "f_session_sin",        # hour-of-day sin encoding
    "f_session_cos",        # hour-of-day cos encoding
    "f_rsi14",              # RSI(14) / 100
    "f_adx14",              # ADX(14) / 100
    "f_spread_norm",        # H-L range / ATR(14)  (spread proxy)
]


def build_features(df: pd.DataFrame, sig_df: pd.DataFrame) -> pd.DataFrame:
    atr14 = atr(df, 14)
    rsi14 = rsi(df["close"], 14)
    adx14 = adx(df, 14)
    hours = df.index.hour

    rows = []
    for _, row in sig_df.iterrows():
        i     = max(0, min(int(row["bar_idx"]), len(df)-1))
        atr_v = max(float(atr14.iloc[i]), 1e-8)
        zw    = row["zone_high"] - row["zone_low"]
        zmid  = (row["zone_high"] + row["zone_low"]) / 2
        hour  = int(hours[i])

        rows.append({
            "f_signal_ob":        int(row["signal_type"] == SIGNAL_OB),
            "f_signal_fvg":       int(row["signal_type"] == SIGNAL_FVG),
            "f_signal_bos":       int(row["signal_type"] == SIGNAL_BOS),
            "f_direction":        float(row["direction"]),
            "f_zone_width_atr":   float(np.clip(zw / atr_v, 0, 10)),
            "f_dist_to_zone_atr": float(np.clip(abs(row["entry_price"] - zmid) / atr_v, 0, 10)),
            "f_fib_pct":          float(np.clip(abs(row["entry_price"] - row["zone_low"]) / max(zw, 1e-8), 0, 1)),
            "f_session_sin":      float(np.sin(2 * np.pi * hour / 24)),
            "f_session_cos":      float(np.cos(2 * np.pi * hour / 24)),
            "f_rsi14":            float(rsi14.iloc[i]) / 100.0,
            "f_adx14":            float(np.clip(adx14.iloc[i], 0, 100)) / 100.0,
            "f_spread_norm":      float(np.clip((df["high"].iloc[i] - df["low"].iloc[i]) / atr_v, 0, 5)),
        })

    feat_df = pd.DataFrame(rows, columns=FEATURE_COLS, index=sig_df.index)
    feat_df["label"] = sig_df["label"].values
    feat_df.dropna(inplace=True)
    print(f"[Feat]  Feature matrix: {feat_df.shape[0]:,} rows x {len(FEATURE_COLS)} features")
    return feat_df


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def train(feat_df: pd.DataFrame, args):
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import classification_report, roc_auc_score
    from xgboost import XGBClassifier

    X = feat_df[FEATURE_COLS].values.astype(np.float32)
    y = feat_df["label"].values.astype(int)

    pos = int(y.sum())
    neg = int(len(y) - pos)
    w   = neg / max(pos, 1)
    print(f"[Train] TP hits: {pos:,}  SL hits: {neg:,}  "
          f"scale_pos_weight: {w:.2f}")

    if len(np.unique(y)) < 2:
        print("[Train] WARNING: only one outcome class — model will be trivial.")
        print("[Train] Fitting anyway for ONNX export. Re-check sl_pts / tp_pts.")
        pipe = Pipeline([("scaler", StandardScaler()),
                         ("xgb", XGBClassifier(n_estimators=100, random_state=42,
                                               use_label_encoder=False,
                                               eval_metric="logloss"))])
        pipe.fit(X, y)
        return pipe

    xgb = XGBClassifier(
        n_estimators   = 400,
        max_depth      = 5,
        learning_rate  = 0.04,
        subsample      = 0.8,
        colsample_bytree = 0.8,
        min_child_weight = 3,
        gamma          = 0.1,
        scale_pos_weight = w,
        eval_metric    = "logloss",
        use_label_encoder = False,
        random_state   = 42,
        n_jobs         = -1,
    )
    pipeline = Pipeline([("scaler", StandardScaler()), ("xgb", xgb)])

    n_splits = min(5, pos, neg)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    auc = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"\n[Train] {n_splits}-Fold CV ROC-AUC: {auc.mean():.4f} +/- {auc.std():.4f}")

    # Interpretation guide printed alongside results
    if   auc.mean() >= 0.70: auc_comment = "Good — model adds real edge"
    elif auc.mean() >= 0.62: auc_comment = "Fair — some predictive signal"
    elif auc.mean() >= 0.55: auc_comment = "Weak — borderline useful"
    else:                    auc_comment = "Poor — near random; check data quality"
    print(f"        Interpretation  : {auc_comment}")

    pipeline.fit(X, y)
    y_pred = pipeline.predict(X)
    y_prob = pipeline.predict_proba(X)[:, 1]

    labels_present = sorted(np.unique(y))
    tgt = ["SL/Expire","TP Hit"] if len(labels_present)==2 else [str(l) for l in labels_present]
    print("\n[Train] In-sample classification report:")
    print(classification_report(y, y_pred, target_names=tgt, labels=labels_present))
    print(f"[Train] In-sample ROC-AUC : {roc_auc_score(y, y_prob):.4f}  "
          "(will be higher than CV — expected due to overfitting)")

    imp = (pd.DataFrame({"feature": FEATURE_COLS,
                          "importance": xgb.feature_importances_})
             .sort_values("importance", ascending=False))
    print("\n[Train] Feature importances (higher = more useful to the model):")
    print(imp.to_string(index=False))

    return pipeline


# ---------------------------------------------------------------------------
# ONNX export + verification
# ---------------------------------------------------------------------------
def export_onnx(pipeline, out_dir: str, mt5_files_path: str = '') -> str:
    from skl2onnx import convert_sklearn, update_registered_converter
    from skl2onnx.common.data_types import FloatTensorType
    from skl2onnx.common.shape_calculator import calculate_linear_classifier_output_shapes
    import onnxruntime as rt

    try:
        from xgboost import XGBClassifier as _XGB
        from onnxmltools.convert.xgboost.operator_converters.XGBoost import convert_xgboost
        update_registered_converter(
            _XGB, "XGBoostXGBClassifier",
            calculate_linear_classifier_output_shapes,
            convert_xgboost,
            options={"nocl": [True, False], "zipmap": [True, False, "columns"]},
        )
    except ImportError:
        sys.exit("[ERROR] onnxmltools required.\n  pip install onnxmltools")

    n_feat = len(FEATURE_COLS)
    onnx_model = convert_sklearn(
        pipeline,
        initial_types=[("float_input", FloatTensorType([None, n_feat]))],
        options={"zipmap": False},
        target_opset={"": 15, "ai.onnx.ml": 3},
    )

    model_bytes = onnx_model.SerializeToString()

    # Always save a local copy
    out_path = os.path.join(out_dir, "smc_filter.onnx")
    with open(out_path, "wb") as f:
        f.write(model_bytes)
    print(f"\n[ONNX] Local copy saved  ->  {out_path}")
    print(f"       File size         :  {Path(out_path).stat().st_size / 1024:.1f} KB")

    # Save directly into MT5 Files folder
    if mt5_files_path and mt5_files_path.strip():
        mt5_dir = Path(mt5_files_path.strip())
        if mt5_dir.exists():
            mt5_dest = mt5_dir / "smc_filter.onnx"
            with open(mt5_dest, "wb") as f:
                f.write(model_bytes)
            print(f"[ONNX] MT5 copy saved    ->  {mt5_dest}")
        else:
            print(f"[ONNX] WARNING: mt5_files_path not found: {mt5_dir}")
            print(f"       Copy \"{out_path}\" to MQL5/Files/ manually.")
    else:
        print("[ONNX] mt5_files_path not set -- copy to MQL5/Files/ manually.")

    # Verify inference works
    sess     = rt.InferenceSession(out_path, providers=["CPUExecutionProvider"])
    dummy    = np.random.rand(1, n_feat).astype(np.float32)
    in_name  = sess.get_inputs()[0].name
    lbl_name = sess.get_outputs()[0].name
    prb_name = sess.get_outputs()[1].name
    lbl, prb = sess.run([lbl_name, prb_name], {in_name: dummy})
    print(f"[ONNX] Verification  ->  label: {lbl}  prob: {prb}")
    print(f"[ONNX] Input shape   :  {sess.get_inputs()[0].shape}")
    print(f"[ONNX] Output names  :  {[o.name for o in sess.get_outputs()]}")

    mt5_path = r"%APPDATA%\MetaQuotes\Terminal\<ID>\MQL5\Files\smc_filter.onnx"
    print(f"\n{'='*55}")
    print(f"  Next steps:")
    print(f"  1. Copy '{out_path}'")
    print(f"     -> {mt5_path}")
    print(f"     (Open MT5: File -> Open Data Folder to find <ID>)")
    print(f"  2. Compile SMC_AI_Filter.mq5 in MetaEditor")
    print(f"  3. Attach EA to XAUUSD H1 chart")
    print(f"  4. Run Strategy Tester on the same date range as your CSV")
    print(f"{'='*55}\n")
    return out_path


# ---------------------------------------------------------------------------
# Audit CSV
# ---------------------------------------------------------------------------
def save_report(feat_df: pd.DataFrame, out_dir: str):
    path = os.path.join(out_dir, "smc_features.csv")
    feat_df.to_csv(path, index=False)
    print(f"[Report] Labelled feature matrix saved -> {path}")
    print(f"         ({len(feat_df):,} rows — open in Excel to inspect signal quality)")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    cfg = get_config()
    Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)

    print("=" * 55)
    print("  SMC AI Filter -- XGBoost Training Pipeline")
    print(f"  Data source : {cfg.data_source.upper()}")
    print(f"  Symbol      : {cfg.symbol}  |  TF: {cfg.timeframe}")
    print(f"  Point size  : {get_point(cfg.symbol)}")
    print(f"  SL: {cfg.sl_pts} pts  |  TP: {cfg.tp_pts} pts")
    print("=" * 55 + "\n")

    # 1. Load data
    src = cfg.data_source
    if src == "csv":
        df = load_csv(cfg.csv_file)
    elif src == "mt5":
        df = load_mt5_bars(cfg.symbol, cfg.timeframe, cfg.bars)
    elif src == "synthetic":
        df = make_synthetic_bars(
            getattr(cfg, "bars", 20000), cfg.symbol, cfg.timeframe)
    else:
        sys.exit(f"[ERROR] Unknown data_source '{src}'. Choose: csv | mt5 | synthetic")

    if len(df) < 500:
        sys.exit(f"[ERROR] Only {len(df)} bars loaded — need at least 500.")

    # 2. Detect SMC signals
    sig_df = detect_signals(df, cfg)
    if len(sig_df) == 0:
        sys.exit("[ERROR] No signals detected. Check symbol/data settings.")

    # 3. Label TP / SL outcomes
    sig_df = label_events(df, sig_df, cfg)
    if len(sig_df) < 100:
        sys.exit(
            f"[ERROR] Only {len(sig_df)} labelled events (need >= 100).\n"
            "  - Extend the date range in fetch_historical_data.py\n"
            "  - Check that sl_pts / tp_pts match the actual price scale"
        )

    # 4. Feature engineering
    feat_df = build_features(df, sig_df)

    # 5. Train
    pipeline = train(feat_df, cfg)

    # 6. Export ONNX
    export_onnx(pipeline, cfg.out_dir, getattr(cfg, 'mt5_files_path', ''))

    # 7. Audit report
    save_report(feat_df, cfg.out_dir)

    print("Pipeline complete.")


if __name__ == "__main__":
    main()

[Config] Jupyter detected -- using NOTEBOOK_CONFIG.
[Config] Edit the NOTEBOOK_CONFIG dict at the top of this file.

  SMC AI Filter -- XGBoost Training Pipeline
  Data source : CSV
  Symbol      : XAUUSD.m  |  TF: H1
  Point size  : 0.01
  SL: 3500 pts  |  TP: 7500 pts

[CSV]   Loaded 13,584 bars from 'XAUUSD_H1.csv'
        Date range : 2023-09-04  ->  2025-12-31
        Price range: 1815.03  ->  4546.69
        Mean H-L   : 8.1656  (8.17 for XAUUSD-style)

[SMC]   Detected 5,019 raw signal events  (OB=1312, FVG=2761, BOS=946)
[Label] Symbol: XAUUSD.m  Point: 0.01
[Label] SL: 3500 pts = 35.0000  |  TP: 7500 pts = 75.0000  |  Lookahead: 2000 bars

[Label] Results:
        Total signals  : 5,019
        Labelled       : 5,014  (99.9% resolved)
        Expired (NaN)  : 5
        Win rate       : 36.8%
        TP hits        : 1,846
        SL hits        : 3,168
        Risk:Reward    : 1:2.14
        Expected win%  : 31.8%  (random baseline)

[Feat]  Feature matrix: 5,014 rows x 12 fea